In [ ]:
# Setup for Google Colab
import os
if not os.path.exists('/content/ML-labs'):
    !git clone https://github.com/pie-12/ML-labs.git /content/ML-labs

os.chdir('/content/ML-labs/')
print("✅ Cấu hình dữ liệu thành công! Thư mục làm việc hiện tại:", os.getcwd())

# Lab 5 - Model Evaluation

**Objective:**
1. Train and evaluate a model on an imbalanced dataset.
2. Use Matplotlib/Seaborn for visualization: Confusion Matrix heatmap, ROC Curve, and Precision-Recall Curve.
3. Understand why Accuracy is misleading for imbalanced data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook')

## 1. Create a Synthetic Imbalanced Dataset

We will generate a dataset with 10,000 samples where 99% belong to class 0 (e.g., normal transactions) and 1% belong to class 1 (e.g., fraudulent transactions).

In [ ]:
# Generate imbalanced dataset
X, y = make_classification(n_samples=10000, n_features=20, n_classes=2, 
                           weights=[0.99, 0.01], random_state=42)

print(f"Total samples: {len(y)}")
print(f"Class 0 (Majority): {sum(y == 0)}")
print(f"Class 1 (Minority/Fraud): {sum(y == 1)}")

## 2. Train and Evaluate Model
Let's split the data and train a simple Logistic Regression model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

### Why Accuracy is Misleading for Imbalanced Data

Let's check the accuracy of our model and also the accuracy of a \"dummy\" model that just predicts the majority class (0) every single time.

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {acc * 100:.2f}%")

dummy_preds = np.zeros_like(y_test)
dummy_acc = accuracy_score(y_test, dummy_preds)
print(f"Dummy Model (Always 0) Accuracy: {dummy_acc * 100:.2f}%")

**Explanation:**
As we can see, simply predicting that every transaction is normal (Class 0) yields a ~99% accuracy because the dataset is so imbalanced. However, this \"dummy\" model fails completely at finding the actual fraudulent transactions (Class 1), which is usually the main goal. Therefore, **Accuracy is an ineffective metric for imbalanced datasets**.

Instead, we should look at the Confusion Matrix, Precision, Recall, and F1-score.

## 3. Confusion Matrix Heatmap

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, 
            xticklabels=['Predicted 0', 'Predicted 1'], 
            yticklabels=['Actual 0', 'Actual 1'])
plt.title('Confusion Matrix', fontsize=16)
plt.show()

print("Classification Report:\n", classification_report(y_test, y_pred))

## 4. ROC Curve

The Receiver Operating Characteristic (ROC) curve plots the True Positive Rate (Recall) against the False Positive Rate (FPR) at various threshold settings.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR / Recall)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=16)
plt.legend(loc="lower right")
plt.show()

## 5. Precision-Recall Curve

For highly imbalanced datasets, the Precision-Recall (PR) curve is often more informative than the ROC curve. It highlights the trade-off between Precision (how many of the predicted positives are real positives) and Recall (how many of the real positives were found).

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
avg_precision = average_precision_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions, color='blue', lw=2, label=f'PR curve (AP = {avg_precision:.2f})')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve', fontsize=16)
plt.legend(loc="lower left")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.show()